# 04 — Create Genie Space for HR Analytics

**J&J HRD 2030 Predictive Hiring Demo**

This notebook provisions a Databricks Genie Space wired to the J&J HR Analytics tables,
configures domain instructions, benchmark questions, and ground-truth SQL queries so that
non-technical HR stakeholders can ask natural-language questions about candidate data.

### What this notebook does
1. Creates a Genie Space via the Databricks REST API
2. Attaches the four analytics tables (`candidates`, `job_requirements`, `training_data`, `candidate_scoring_summary`) and the `hiring_analytics` metric view
3. Configures 5 benchmark sample questions and detailed domain instructions
4. Registers 2 ground-truth SQL queries for Genie benchmarking
5. Sends a test question through the Genie conversation API and prints the response

### Prerequisites
- Notebooks `00` – `03` have been run (Silver tables + Gold summary + metric view exist)
- A SQL Warehouse ID is provided via the `warehouse_id` widget
- The running user has `CAN USE` permission on the warehouse

**Next notebook:** `05_ml_model.ipynb`

## Setup

Install dependencies, configure widgets, and resolve workspace auth and SQL warehouse.

In [ ]:
%pip install databricks-sdk -q

In [ ]:
# ---------------------------------------------------------------------------
# Widget definitions — edit defaults here or override at runtime via the UI
# ---------------------------------------------------------------------------
dbutils.widgets.text("catalog",      "bx4",      "UC Catalog")
dbutils.widgets.text("schema",       "hrd_2030", "UC Schema")
dbutils.widgets.text("warehouse_id", "",         "SQL Warehouse ID")

In [ ]:
# ---------------------------------------------------------------------------
# Imports and workspace context
# ---------------------------------------------------------------------------
import json, os, requests

catalog      = dbutils.widgets.get("catalog")
schema       = dbutils.widgets.get("schema")
warehouse_id = dbutils.widgets.get("warehouse_id")

# Auth — works in job context and interactive notebooks
w_ctx  = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
host   = w_ctx.apiUrl().get().rstrip("/")
token  = w_ctx.apiToken().get()
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

# Auto-detect warehouse if not provided
if not warehouse_id:
    from databricks.sdk import WorkspaceClient
    _wc = WorkspaceClient()
    _warehouses = list(_wc.warehouses.list())
    if _warehouses:
        warehouse_id = _warehouses[0].id
        print(f"Auto-selected warehouse: {_warehouses[0].name} ({warehouse_id})")
    else:
        raise ValueError("No SQL warehouses found. Provide warehouse_id widget.")

print(f"Catalog      : {catalog}")
print(f"Schema       : {schema}")
print(f"Warehouse ID : {warehouse_id}")
print(f"Host         : {host}")

## Load Config & Create Genie Space

Read the Genie Space definition from `04b_genie_space.json`, substitute catalog/schema placeholders, then create or update the space via the REST API.

In [ ]:
# ---------------------------------------------------------------------------
# Load 04b_genie_space.json (same directory as this notebook) and create/update space
# ---------------------------------------------------------------------------
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
notebook_dir  = os.path.dirname(notebook_path)
genie_json_ws_path = f"/Workspace{notebook_dir}/04b_genie_space.json"

serialized_space_str = ""
genie_meta = {}
if os.path.exists(genie_json_ws_path):
    with open(genie_json_ws_path) as _f:
        genie_meta = json.load(_f)
    # serialized_space is a JSON object in the file — stringify and replace placeholders
    ss_raw = json.dumps(genie_meta.get("serialized_space", {}))
    ss_raw = ss_raw.replace("__CATALOG__", catalog).replace("__SCHEMA__", schema)
    serialized_space_str = ss_raw
    print(f"✅ Loaded 04b_genie_space.json from {genie_json_ws_path}")
else:
    print(f"⚠️  04b_genie_space.json not found at {genie_json_ws_path}")

payload = {
    "warehouse_id":     warehouse_id,
    "title":            genie_meta.get("title", "J&J HRD 2030 — Hiring Analytics Genie"),
    "description":      genie_meta.get("description", ""),
    "sample_questions": genie_meta.get("sample_questions", []),
}
if serialized_space_str:
    payload["serialized_space"] = serialized_space_str

genie_space_id = ""

# Check if an HRD 2030 space already exists
_list_resp = requests.get(f"{host}/api/2.0/genie/spaces", headers=headers, timeout=30)
if _list_resp.status_code == 200:
    for _s in _list_resp.json().get("spaces", []):
        _t = _s.get("title", "")
        if "HRD 2030" in _t or "Hiring Analytics Genie" in _t:
            genie_space_id = _s["space_id"]
            print(f"✅ Found existing space: {_t} ({genie_space_id}) — updating...")
            break

if genie_space_id:
    resp = requests.patch(
        f"{host}/api/2.0/genie/spaces/{genie_space_id}",
        headers=headers, json=payload, timeout=30
    )
    if resp.status_code == 200:
        print(f"✅ Updated Genie Space: {genie_space_id}")
    else:
        print(f"⚠️  Update failed ({resp.status_code}): {resp.text[:300]}")
        genie_space_id = ""

if not genie_space_id:
    print("Creating new Genie Space...")
    resp = requests.post(f"{host}/api/2.0/genie/spaces", headers=headers, json=payload, timeout=30)
    print(f"  HTTP {resp.status_code}: {resp.text[:300]}")
    if resp.status_code in (200, 201):
        genie_space_id = resp.json().get("space_id", "")
        print(f"✅ Created: {genie_space_id}")
    else:
        print("⚠️  Could not create Genie Space.")

print()
print("=" * 60)
print(f"  GENIE SPACE ID: {genie_space_id}")
print("=" * 60)

dbutils.jobs.taskValues.set(key="genie_space_id", value=genie_space_id)

## Register Ground-Truth SQL

Register curated SQL queries as verified answers to benchmark questions so Genie returns consistent, accurate results.

In [ ]:
# ---------------------------------------------------------------------------
# Register ground-truth SQL queries (only if space was created)
# ---------------------------------------------------------------------------
if not genie_space_id:
    print("Skipping ground-truth registration — no Genie Space ID.")
else:
    ground_truth_queries = [
        {
            "question": "Who are the top 5 candidates by total score?",
            "sql": (
                f"SELECT candidate_id, first_name, last_name, job_id, "
                f"total_score, hired "
                f"FROM {catalog}.{schema}.candidate_scoring_summary "
                f"WHERE total_score IS NOT NULL "
                f"ORDER BY total_score DESC LIMIT 5"
            ),
        },
        {
            "question": "What is the hire rate by education level?",
            "sql": (
                f"SELECT education_level, COUNT(*) AS total_candidates, "
                f"SUM(hired) AS hired_count, "
                f"ROUND(AVG(hired) * 100, 1) AS hire_rate_pct "
                f"FROM {catalog}.{schema}.candidate_scoring_summary "
                f"WHERE hired IS NOT NULL "
                f"GROUP BY education_level ORDER BY hire_rate_pct DESC"
            ),
        },
    ]

    instructions_endpoint = f"{host}/api/2.0/genie/spaces/{genie_space_id}/instructions"
    for qt in ground_truth_queries:
        try:
            payload = {"question": qt["question"], "verified_sql": qt["sql"]}
            resp = requests.post(instructions_endpoint, headers=headers, json=payload)
            if resp.status_code in (200, 201):
                print(f"✅ Ground-truth registered: {qt['question'][:60]}...")
            else:
                print(f"⚠️  HTTP {resp.status_code}: {resp.text[:200]}")
        except Exception as e:
            print(f"⚠️  {e}")

print("Ground-truth step complete.")

## Test the Genie Space

Send a sample natural-language question to the Genie conversation API and poll until it returns a response.

In [ ]:
# ---------------------------------------------------------------------------
# Test the Genie Space with a sample question (only if space was created)
# ---------------------------------------------------------------------------
if not genie_space_id:
    print("Skipping conversation test — no Genie Space ID.")
    print("\u2705 Genie Space notebook complete.")
else:
    test_question = "Who are the top candidates for the Director of HR role?"
    print(f"Sending test question: '{test_question}'")

    try:
        conv_resp = requests.post(
            f"{host}/api/2.0/genie/spaces/{genie_space_id}/start-conversation",
            headers=headers,
            json={"content": test_question},
        )

        if conv_resp.status_code != 200:
            print(f"\u26a0\ufe0f  Could not start conversation (HTTP {conv_resp.status_code}): {conv_resp.text[:300]}")
        else:
            conv_data       = conv_resp.json()
            conversation_id = conv_data.get("conversation_id") or conv_data.get("conversation", {}).get("id")
            message_id      = conv_data.get("message_id") or conv_data.get("message", {}).get("id")

            poll_url = (
                f"{host}/api/2.0/genie/spaces/{genie_space_id}"
                f"/conversations/{conversation_id}/messages/{message_id}"
            )
            max_wait, waited = 60, 0
            result = None
            while waited < max_wait:
                poll_resp = requests.get(poll_url, headers=headers)
                if poll_resp.status_code != 200:
                    break
                msg    = poll_resp.json()
                status = msg.get("status", "UNKNOWN")
                print(f"  [{waited:>2}s] Status: {status}")
                if status in ("COMPLETED", "FAILED", "CANCELLED"):
                    result = msg
                    break
                time.sleep(5)
                waited += 5

            if result:
                for att in result.get("attachments", []):
                    if "text" in att:
                        print(att["text"].get("content", ""))
                    elif "query" in att:
                        print("SQL:", att["query"].get("query", ""))
    except Exception as e:
        print(f"\u26a0\ufe0f  Test conversation error: {e}")

    print(f"\n\u2705 Genie Space test complete. Space ID: {genie_space_id}")

## Summary

The Genie Space has been created and is ready for HR stakeholders to use.

| Item | Value |
|---|---|
| Space title | J&J HRD 2030 — Hiring Analytics Genie |
| Tables attached | `candidates`, `job_requirements`, `training_data`, `candidate_scoring_summary`, `hiring_analytics` |
| Sample questions | 5 configured |
| Ground-truth queries | 2 registered |

The `genie_space_id` has been stored as a Databricks task value and can be retrieved in downstream
notebooks or the agent using:

```python
genie_space_id = dbutils.jobs.taskValues.get(taskKey="genie_space", key="genie_space_id")
```

> **Note for the agent notebook (`06_evaluate_register_agent.ipynb`):** The Genie Space ID above is required when
> configuring the `GenieSpaceTool` so that the HireRight agent can delegate natural-language
> HR analytics queries to Genie.

**Proceed to** `05_train_ml_model.ipynb` to train and deploy the predictive hiring model.